# Example 3.7.2 / 1: Save and restore a session

Companion notebook for **3.7.2 Warm Starts and Snapshots** in the AMPL lab Documentation, section "Save and restore a session". It runs the page's complete program: it solves the production model, takes a snapshot, rebuilds the session in a second `AMPL` object, and solves a changed scenario there. Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Save and restore a session

With no `filename`, `snapshot()` returns the session as AMPL text; `eval()` rebuilds it in a new `AMPL` object.

In [2]:
from amplpy import AMPL

ampl = AMPL()
ampl.eval("""
set PROD;
param rate {PROD} > 0;
param profit {PROD};
param market {PROD} >= 0;
param avail >= 0;
var Make {p in PROD} >= 0, <= market[p];
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} Make[p] / rate[p] <= avail;
""")
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40
ampl.solve(solver="highs", verbose=False)
assert ampl.solve_result == "solved"
saved = ampl.snapshot()

continued = AMPL()
continued.eval(saved)
print(continued.var["Make"].to_dict())
continued.param["avail"] = 45
continued.solve(solver="highs", verbose=False)
print(continued.solve_result, continued.obj["Total_Profit"].value())

{'bands': 6000, 'coils': 1400}
solved 213000.0
